# State-of-the-Art Deepfake Detection Model

## Training on 140k Real and Fake Faces Dataset

This notebook trains a state-of-the-art deepfake detection model using:
- **ConvNeXt Large** (SOTA CNN architecture)
- **Advanced augmentation** (MixUp, CutMix, Multi-scale)
- **Focal Loss** for class imbalance
- **MTCNN** for high-quality face detection
- **Optimized for Kaggle GPU**

**Dataset**: 140k Real and Fake Faces  
**Expected Performance**: 92-96% Accuracy, 0.96-0.99 AUC-ROC


In [ ]:
# Cell 1: Setup and Imports
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
import numpy as np
from PIL import Image
import cv2
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, classification_report
from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import json
import random
import warnings
warnings.filterwarnings('ignore')

# Install required packages
try:
    import timm
except:
    print("Installing timm...")
    os.system("pip install timm")
    import timm

try:
    from facenet_pytorch import MTCNN
    USE_MTCNN = True
except:
    print("Installing facenet-pytorch for better face detection...")
    os.system("pip install facenet-pytorch")
    from facenet_pytorch import MTCNN
    USE_MTCNN = True

# Configuration - STATE-OF-THE-ART SETTINGS
CONFIG = {
    'image_size': 224,
    'batch_size': 32,  # Adjust based on GPU memory (T4: 32, P100: 64, V100: 128)
    'num_epochs': 50,
    'learning_rate': 1e-4,
    'weight_decay': 1e-4,
    'num_workers': 4,
    'device': 'cuda' if torch.cuda.is_available() else 'cpu',
    
    # SOTA Model Options (ranked by performance):
    # 1. 'convnext_large' - Best CNN architecture (RECOMMENDED - Best balance)
    # 2. 'vit_large_patch16_224' - Vision Transformer Large (Best accuracy, slower)
    # 3. 'swin_large_patch4_window7_224' - Swin Transformer (Good balance)
    # 4. 'efficientnetv2_xl' - EfficientNet V2 XL (Faster, good accuracy)
    'model_name': 'convnext_large',  # CURRENT SOTA for deepfake detection
    
    # Dataset paths (matches your Kaggle dataset structure)
    'data_root': '/kaggle/input/140k-real-and-fake-faces',
    'train_path': '/kaggle/input/140k-real-and-fake-faces/real_vs_fake/real-vs-fake/train',
    'valid_path': '/kaggle/input/140k-real-and-fake-faces/real_vs_fake/real-vs-fake/valid',
    'test_path': '/kaggle/input/140k-real-and-fake-faces/real_vs_fake/real-vs-fake/test',
    
    'save_path': '/kaggle/working/best_deepfake_model.pth',
    
    # Advanced Training Techniques
    'use_face_crop': True,  # Your model was trained on face crops
    'use_mixup': True,
    'mixup_alpha': 0.4,
    'use_cutmix': True,
    'cutmix_alpha': 1.0,
    'label_smoothing': 0.1,
    'use_focal_loss': True,
    'focal_alpha': 0.25,
    'focal_gamma': 2.0,
    'use_multi_scale': True,
    'multi_scale_sizes': [224, 256, 288],
}

print(f"✅ Device: {CONFIG['device']}")
print(f"✅ Model: {CONFIG['model_name']} (SOTA Architecture)")
print(f"✅ Advanced Techniques: Mixup={CONFIG['use_mixup']}, CutMix={CONFIG['use_cutmix']}, Focal Loss={CONFIG['use_focal_loss']}")


In [ ]:
# Cell 2: Advanced Face Detection with MTCNN
print("\n🔍 Initializing Face Detector...")
mtcnn = MTCNN(image_size=224, margin=40, device=CONFIG['device'])
print("✅ MTCNN face detector loaded (best quality)")

# OpenCV fallback
face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')

def detect_and_crop_face_advanced(image):
    """Advanced face detection with MTCNN or OpenCV fallback"""
    try:
        # MTCNN returns (image, prob, landmarks)
        face_img, prob = mtcnn(image, return_prob=True)
        if face_img is not None and prob > 0.9:
            # Convert tensor to PIL
            face_array = face_img.permute(1, 2, 0).cpu().numpy()
            face_array = (face_array * 255).astype(np.uint8)
            face_array = np.clip(face_array, 0, 255)
            return Image.fromarray(face_array)
    except Exception as e:
        pass
    
    # OpenCV fallback
    try:
        img_array = np.array(image)
        if len(img_array.shape) == 2:
            img_array = cv2.cvtColor(img_array, cv2.COLOR_GRAY2RGB)
        
        gray = cv2.cvtColor(img_array, cv2.COLOR_RGB2GRAY)
        faces = face_cascade.detectMultiScale(gray, 1.3, 5)
        
        if len(faces) > 0:
            x, y, w, h = faces[0]
            # Add padding
            padding = int(w * 0.3)
            x1 = max(0, x - padding)
            y1 = max(0, y - padding)
            x2 = min(img_array.shape[1], x + w + padding)
            y2 = min(img_array.shape[0], y + h + padding)
            return image.crop((x1, y1, x2, y2))
    except:
        pass
    
    # Center crop fallback
    w, h = image.size
    size = min(w, h)
    left = (w - size) // 2
    top = (h - size) // 2
    return image.crop((left, top, left + size, top + size))


In [ ]:
# Cell 3: Advanced Data Augmentation Classes
class MixUp:
    """MixUp augmentation"""
    def __init__(self, alpha=0.4):
        self.alpha = alpha
    
    def __call__(self, batch):
        if random.random() > 0.5:
            return batch
        
        images, labels = batch
        batch_size = images.size(0)
        indices = torch.randperm(batch_size)
        
        lam = np.random.beta(self.alpha, self.alpha)
        mixed_images = lam * images + (1 - lam) * images[indices]
        y_a, y_b = labels, labels[indices]
        
        return mixed_images, y_a, y_b, lam

class CutMix:
    """CutMix augmentation"""
    def __init__(self, alpha=1.0):
        self.alpha = alpha
    
    def __call__(self, batch):
        if random.random() > 0.5:
            return batch
        
        images, labels = batch
        batch_size = images.size(0)
        indices = torch.randperm(batch_size)
        
        lam = np.random.beta(self.alpha, self.alpha)
        bbx1, bby1, bbx2, bby2 = self.rand_bbox(images.size(), lam)
        images[:, :, bbx1:bbx2, bby1:bby2] = images[indices, :, bbx1:bbx2, bby1:bby2]
        
        # Adjust lambda to match pixel ratio
        lam = 1 - ((bbx2 - bbx1) * (bby2 - bby1) / (images.size()[-1] * images.size()[-2]))
        y_a, y_b = labels, labels[indices]
        
        return images, y_a, y_b, lam
    
    def rand_bbox(self, size, lam):
        W = size[2]
        H = size[3]
        cut_rat = np.sqrt(1. - lam)
        cut_w = np.int(W * cut_rat)
        cut_h = np.int(H * cut_rat)
        
        cx = np.random.randint(W)
        cy = np.random.randint(H)
        
        bbx1 = np.clip(cx - cut_w // 2, 0, W)
        bby1 = np.clip(cy - cut_h // 2, 0, H)
        bbx2 = np.clip(cx + cut_w // 2, 0, W)
        bby2 = np.clip(cy + cut_h // 2, 0, H)
        
        return bbx1, bby1, bbx2, bby2


In [ ]:
# Cell 4: Advanced Loss Functions
class FocalLoss(nn.Module):
    """Focal Loss for handling class imbalance"""
    def __init__(self, alpha=0.25, gamma=2.0, reduction='mean'):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.reduction = reduction
    
    def forward(self, inputs, targets):
        ce_loss = F.cross_entropy(inputs, targets, reduction='none')
        pt = torch.exp(-ce_loss)
        focal_loss = self.alpha * (1 - pt) ** self.gamma * ce_loss
        
        if self.reduction == 'mean':
            return focal_loss.mean()
        elif self.reduction == 'sum':
            return focal_loss.sum()
        return focal_loss

class LabelSmoothingCrossEntropy(nn.Module):
    """Label smoothing for better generalization"""
    def __init__(self, smoothing=0.1):
        super().__init__()
        self.smoothing = smoothing
    
    def forward(self, pred, target):
        log_prob = F.log_softmax(pred, dim=-1)
        nll_loss = -log_prob.gather(dim=-1, index=target.unsqueeze(1)).squeeze(1)
        smooth_loss = -log_prob.mean(dim=-1)
        loss = (1 - self.smoothing) * nll_loss + self.smoothing * smooth_loss
        return loss.mean()


In [ ]:
# Cell 5: Dataset Class (Optimized for your dataset structure)
class DeepfakeDataset(Dataset):
    def __init__(self, data_dir, transform=None, use_face_crop=True, is_training=False):
        self.data_dir = Path(data_dir)
        self.transform = transform
        self.use_face_crop = use_face_crop
        self.is_training = is_training
        
        # Your dataset structure: train/real/ and train/fake/
        real_dir = self.data_dir / 'real'
        fake_dir = self.data_dir / 'fake'
        
        self.image_paths = []
        self.labels = []
        
        # Real images (label 0)
        if real_dir.exists():
            real_images = (list(real_dir.glob('*.jpg')) + 
                          list(real_dir.glob('*.png')) + 
                          list(real_dir.glob('*.jpeg')) + 
                          list(real_dir.glob('*.JPG')))
            self.image_paths.extend(real_images)
            self.labels.extend([0] * len(real_images))
        
        # Fake images (label 1)
        if fake_dir.exists():
            fake_images = (list(fake_dir.glob('*.jpg')) + 
                          list(fake_dir.glob('*.png')) + 
                          list(fake_dir.glob('*.jpeg')) + 
                          list(fake_dir.glob('*.JPG')))
            self.image_paths.extend(fake_images)
            self.labels.extend([1] * len(fake_images))
        
        if len(self.image_paths) == 0:
            raise ValueError(f"No images found in {data_dir}")
        
        print(f"   Loaded {len(self.image_paths):,} images from {self.data_dir.name}")
        print(f"      Real: {sum(1 for l in self.labels if l == 0):,}")
        print(f"      Fake: {sum(1 for l in self.labels if l == 1):,}")
    
    def __len__(self):
        return len(self.image_paths)
    
    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        label = self.labels[idx]
        
        # Load image with error handling
        max_retries = 3
        for retry in range(max_retries):
            try:
                image = Image.open(img_path).convert('RGB')
                break
            except Exception as e:
                if retry == max_retries - 1:
                    # Return black image as fallback
                    image = Image.new('RGB', (224, 224), (0, 0, 0))
                else:
                    # Try next image
                    idx = (idx + 1) % len(self.image_paths)
                    img_path = self.image_paths[idx]
                    label = self.labels[idx]
        
        # Face detection and cropping
        if self.use_face_crop:
            image = detect_and_crop_face_advanced(image)
        
        # Multi-scale training
        if self.is_training and CONFIG['use_multi_scale']:
            size = random.choice(CONFIG['multi_scale_sizes'])
            # Create temporary transform with new size
            temp_transform = transforms.Compose([
                transforms.Resize((size, size)),
                transforms.ToTensor(),
                transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
            ])
            image = temp_transform(image)
        else:
            if self.transform:
                image = self.transform(image)
            else:
                image = transforms.Resize((224, 224))(image)
                image = transforms.ToTensor()(image)
        
        return image, label


In [ ]:
# Cell 6: Data Transforms
train_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomCrop(224),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.1),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.1),
    transforms.RandomRotation(15),
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1), scale=(0.9, 1.1)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    transforms.RandomErasing(p=0.2, scale=(0.02, 0.33)),
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])


In [ ]:
# Cell 7: Load Datasets (Using your pre-split train/valid/test)
print("\n📂 Loading Datasets...")
train_dataset = DeepfakeDataset(
    CONFIG['train_path'],
    transform=train_transform,
    use_face_crop=CONFIG['use_face_crop'],
    is_training=True
)

valid_dataset = DeepfakeDataset(
    CONFIG['valid_path'],
    transform=val_transform,
    use_face_crop=CONFIG['use_face_crop'],
    is_training=False
)

test_dataset = DeepfakeDataset(
    CONFIG['test_path'],
    transform=val_transform,
    use_face_crop=CONFIG['use_face_crop'],
    is_training=False
)

train_loader = DataLoader(
    train_dataset, batch_size=CONFIG['batch_size'],
    shuffle=True, num_workers=CONFIG['num_workers'], pin_memory=True, drop_last=True
)

valid_loader = DataLoader(
    valid_dataset, batch_size=CONFIG['batch_size'],
    shuffle=False, num_workers=CONFIG['num_workers'], pin_memory=True
)

test_loader = DataLoader(
    test_dataset, batch_size=CONFIG['batch_size'],
    shuffle=False, num_workers=CONFIG['num_workers'], pin_memory=True
)

print(f"\n✅ Dataset Summary:")
print(f"   Train: {len(train_dataset):,} images")
print(f"   Valid: {len(valid_dataset):,} images")
print(f"   Test:  {len(test_dataset):,} images")


In [ ]:
# Cell 8: Create SOTA Model
from timm import create_model

def create_sota_model(model_name='convnext_large', num_classes=2):
    """Create state-of-the-art model using timm"""
    try:
        # Use timm to create model with pretrained weights
        model = create_model(
            model_name,
            pretrained=True,
            num_classes=num_classes,
            drop_rate=0.3,
            drop_path_rate=0.2
        )
        print(f"✅ Created {model_name} with pretrained weights")
        return model
    except Exception as e:
        print(f"⚠️ Error creating {model_name}: {e}")
        print("Falling back to EfficientNet-B4")
        from torchvision.models import efficientnet_b4, EfficientNet_B4_Weights
        model = efficientnet_b4(weights=EfficientNet_B4_Weights.IMAGENET1K_V1)
        model.classifier = nn.Sequential(
            nn.Dropout(0.4),
            nn.Linear(model.classifier[1].in_features, num_classes)
        )
        return model

print("\n🏗️  Creating Model...")
model = create_sota_model(CONFIG['model_name'], num_classes=2)
model = model.to(CONFIG['device'])

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"✅ Model: {total_params:,} total params, {trainable_params:,} trainable")


In [ ]:
# Cell 9: Loss Function Setup
if CONFIG['use_focal_loss']:
    criterion = FocalLoss(alpha=CONFIG['focal_alpha'], gamma=CONFIG['focal_gamma'])
    print("✅ Using Focal Loss")
elif CONFIG['label_smoothing'] > 0:
    criterion = LabelSmoothingCrossEntropy(smoothing=CONFIG['label_smoothing'])
    print("✅ Using Label Smoothing")
else:
    criterion = nn.CrossEntropyLoss()
    print("✅ Using Standard CrossEntropy Loss")

# Initialize augmentation
mixup = MixUp(alpha=CONFIG['mixup_alpha']) if CONFIG['use_mixup'] else None
cutmix = CutMix(alpha=CONFIG['cutmix_alpha']) if CONFIG['use_cutmix'] else None


In [ ]:
# Cell 10: Optimizer and Scheduler
optimizer = optim.AdamW(
    model.parameters(),
    lr=CONFIG['learning_rate'],
    weight_decay=CONFIG['weight_decay'],
    betas=(0.9, 0.999)
)

# Cosine annealing with warm restarts
scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(
    optimizer, T_0=10, T_mult=2, eta_min=1e-6
)

print("✅ Optimizer and scheduler configured")


In [ ]:
# Cell 11: Training Loop with Advanced Techniques
def train_epoch_advanced(model, loader, criterion, optimizer, device, epoch):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    
    pbar = tqdm(loader, desc=f'Epoch {epoch+1} [Train]')
    for batch_idx, (images, labels) in enumerate(pbar):
        images, labels = images.to(device), labels.to(device)
        
        # Apply MixUp or CutMix
        if CONFIG['use_mixup'] and random.random() < 0.5:
            mixed_images, y_a, y_b, lam = mixup((images, labels))
            optimizer.zero_grad()
            outputs = model(mixed_images)
            loss = lam * criterion(outputs, y_a) + (1 - lam) * criterion(outputs, y_b)
        elif CONFIG['use_cutmix'] and random.random() < 0.5:
            mixed_images, y_a, y_b, lam = cutmix((images, labels))
            optimizer.zero_grad()
            outputs = model(mixed_images)
            loss = lam * criterion(outputs, y_a) + (1 - lam) * criterion(outputs, y_b)
        else:
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
        
        loss.backward()
        # Gradient clipping
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        
        running_loss += loss.item()
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()
        
        pbar.set_postfix({
            'loss': f'{running_loss/(batch_idx+1):.4f}',
            'acc': f'{100.*correct/total:.2f}%',
            'lr': f'{optimizer.param_groups[0]["lr"]:.2e}'
        })
    
    epoch_loss = running_loss / len(loader)
    epoch_acc = 100. * correct / total
    return epoch_loss, epoch_acc

def validate(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for images, labels in tqdm(loader, desc='Validating'):
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            running_loss += loss.item()
            probs = torch.softmax(outputs, dim=1)
            all_preds.extend(probs[:, 1].cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    
    epoch_loss = running_loss / len(loader)
    all_preds = np.array(all_preds)
    all_labels = np.array(all_labels)
    all_preds_binary = (all_preds > 0.5).astype(int)
    
    # Calculate metrics
    acc = accuracy_score(all_labels, all_preds_binary)
    precision = precision_score(all_labels, all_preds_binary, zero_division=0)
    recall = recall_score(all_labels, all_preds_binary, zero_division=0)
    f1 = f1_score(all_labels, all_preds_binary, zero_division=0)
    auc = roc_auc_score(all_labels, all_preds)
    
    return epoch_loss, acc, precision, recall, f1, auc


In [ ]:
# Cell 12: Main Training Loop
best_val_auc = 0.0
best_val_acc = 0.0
train_history = {
    'loss': [], 'acc': [], 'val_loss': [], 'val_acc': [],
    'val_precision': [], 'val_recall': [], 'val_f1': [], 'val_auc': []
}

print("\n" + "="*70)
print("🚀 STARTING STATE-OF-THE-ART TRAINING")
print("="*70)

for epoch in range(CONFIG['num_epochs']):
    print(f"\n{'='*70}")
    print(f"Epoch {epoch+1}/{CONFIG['num_epochs']}")
    print(f"{'='*70}")
    
    # Train
    train_loss, train_acc = train_epoch_advanced(
        model, train_loader, criterion, optimizer, CONFIG['device'], epoch
    )
    
    # Validate
    val_loss, val_acc, val_precision, val_recall, val_f1, val_auc = validate(
        model, valid_loader, criterion, CONFIG['device']
    )
    
    # Update learning rate
    scheduler.step()
    
    # Save history
    train_history['loss'].append(train_loss)
    train_history['acc'].append(train_acc)
    train_history['val_loss'].append(val_loss)
    train_history['val_acc'].append(val_acc)
    train_history['val_precision'].append(val_precision)
    train_history['val_recall'].append(val_recall)
    train_history['val_f1'].append(val_f1)
    train_history['val_auc'].append(val_auc)
    
    # Save best model (based on AUC)
    if val_auc > best_val_auc:
        best_val_auc = val_auc
        best_val_acc = val_acc
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_auc': val_auc,
            'val_acc': val_acc,
            'val_precision': val_precision,
            'val_recall': val_recall,
            'val_f1': val_f1,
            'config': CONFIG,
            'history': train_history
        }, CONFIG['save_path'])
        print(f"✅ Saved best model (AUC: {val_auc:.4f}, Acc: {val_acc:.4f})")
    
    print(f"\nTrain - Loss: {train_loss:.4f}, Acc: {train_acc:.2f}%")
    print(f"Val   - Loss: {val_loss:.4f}, Acc: {val_acc:.4f}, AUC: {val_auc:.4f}")
    print(f"Val   - Precision: {val_precision:.4f}, Recall: {val_recall:.4f}, F1: {val_f1:.4f}")

print("\n✅ Training Complete!")


In [ ]:
# Cell 13: Final Test Evaluation
print("\n" + "="*70)
print("📊 FINAL TEST SET EVALUATION")
print("="*70)

checkpoint = torch.load(CONFIG['save_path'])
model.load_state_dict(checkpoint['model_state_dict'])

test_loss, test_acc, test_precision, test_recall, test_f1, test_auc = validate(
    model, test_loader, criterion, CONFIG['device']
)

print(f"\n🎯 Test Set Results:")
print(f"   Accuracy:  {test_acc:.4f} ({test_acc*100:.2f}%)")
print(f"   Precision: {test_precision:.4f}")
print(f"   Recall:    {test_recall:.4f}")
print(f"   F1-Score:  {test_f1:.4f}")
print(f"   AUC-ROC:   {test_auc:.4f}")

# Confusion Matrix
model.eval()
all_preds = []
all_labels = []
with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(CONFIG['device'])
        outputs = model(images)
        _, preds = outputs.max(1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.numpy())

cm = confusion_matrix(all_labels, all_preds)
print(f"\n📊 Confusion Matrix:")
print(f"   [[TN={cm[0,0]}, FP={cm[0,1]}],")
print(f"    [FN={cm[1,0]}, TP={cm[1,1]}]]")

# Save final model
final_model_path = '/kaggle/working/deepfake_detection_sota.pth'
torch.save({
    'model_state_dict': model.state_dict(),
    'config': CONFIG,
    'test_metrics': {
        'accuracy': test_acc,
        'precision': test_precision,
        'recall': test_recall,
        'f1': test_f1,
        'auc': test_auc
    }
}, final_model_path)

print(f"\n✅ Model saved: {final_model_path}")
print("📥 Download this file and update your backend!")
